# Surveillance System - Demo
Detectez des personnes dans votre video avec YOLOv8

In [ ]:
!pip install ultralytics==8.3.0 supervision==0.20.0 -q
print('Packages installes !')

In [ ]:
from google.colab import files
print('Upload ta video MP4 :')
uploaded = files.upload()
video_name = list(uploaded.keys())[0]
print(f'Video chargee : {video_name}')

In [ ]:
import cv2, numpy as np, supervision as sv, time
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
tracker = sv.ByteTrack()
box_annotator = sv.BoundingBoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5)
trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=40)

output_path = 'resultat_annote.mp4'
cap = cv2.VideoCapture(video_name)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
line_counter = sv.LineZone(start=sv.Point(w//2, 0), end=sv.Point(w//2, h))

frame_count, ids_vus, debut = 0, set(), time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, conf=0.5, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[detections.class_id == 0]
    detections = tracker.update_with_detections(detections)
    tracker_ids = detections.tracker_id if detections.tracker_id is not None else np.array([])
    ids_vus.update(tracker_ids.tolist())
    line_counter.trigger(detections)
    labels = [f'#{tid}' for tid in tracker_ids]
    annotated = trace_annotator.annotate(frame.copy(), detections)
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels)
    cv2.putText(annotated, f'Personnes: {len(detections)}', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    writer.write(annotated)
    frame_count += 1

cap.release()
writer.release()
duree = time.time() - debut

print(f'Frames traitees   : {frame_count}')
print(f'Personnes uniques : {len(ids_vus)}')
print(f'Entrees           : {line_counter.in_count}')
print(f'Sorties           : {line_counter.out_count}')
print(f'FPS traitement    : {round(frame_count/duree, 1)}')

In [ ]:
from google.colab import files
files.download('resultat_annote.mp4')
print('Video annotee telechargee !')